# KolinAI — картинки без цензуры на бесплатной GPU Google

1. **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**
2. **Среда выполнения → Выполнить все** (Run all)
3. Подожди 3-8 минут. Внизу появится ссылка вида `https://xxxxx.gradio.live` — открой её.
4. Выбирай модель, пиши промт, жми **Сгенерировать**.

Бесплатная GPU: несколько часов в день (Colab). Если лимит — тот же файл работает в Kaggle (30 ч/нед): kaggle.com → Create → New Notebook → File → Import notebook.

Модели: **Z-Image Turbo** (реализм, 8 шагов, без встроенного фильтра, негатив не работает) и **WAI NSFW Illustrious v1.5** (аниме 18+, негатив работает).

In [ ]:
# Ставим свежие diffusers и gradio (torch уже есть в Colab)
!pip install -q -U diffusers transformers accelerate safetensors gradio
print("Установка готова. Запускай следующую ячейку (Ctrl+Enter).")


In [ ]:
import gc, random, torch, gradio as gr
from diffusers import DiffusionPipeline

MODELS = {
    "Z-Image Turbo — реализм, быстро (негатив не работает, CFG=1)":
        dict(repo="Tongyi-MAI/Z-Image-Turbo", steps=8, cfg=1.0, neg=False),
    "WAI NSFW Illustrious v1.5 — аниме 18+":
        dict(repo="John6666/wai-nsfw-illustrious-sdxl-v150-sdxl", steps=26, cfg=5.0, neg=True),
}
state = {"key": None, "pipe": None}

def load_pipe(key):
    if state["key"] == key:
        return state["pipe"]
    if state["pipe"] is not None:
        del state["pipe"]; gc.collect(); torch.cuda.empty_cache()
    pipe = DiffusionPipeline.from_pretrained(
        MODELS[key]["repo"], torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    state["key"], state["pipe"] = key, pipe
    return pipe

def gen(key, prompt, neg, steps, cfg, w, h, seed):
    if not prompt or not prompt.strip():
        raise gr.Error("Напиши промт")
    m = MODELS[key]
    pipe = load_pipe(key)
    if int(seed) < 0:
        seed = random.randint(0, 2**31 - 1)
    g = torch.Generator("cuda").manual_seed(int(seed))
    img = pipe(
        prompt=prompt,
        negative_prompt=(neg if m["neg"] else None),
        num_inference_steps=int(steps),
        guidance_scale=(cfg if m["neg"] else 1.0),
        width=int(w), height=int(h), generator=g,
    ).images[0]
    return img, "seed: %d" % int(seed)

with gr.Blocks(title="KolinAI Colab") as demo:
    gr.Markdown(
        "## KolinAI — картинки на бесплатной GPU\n"
        "Первая генерация качает модель (до 3-5 минут), дальше быстро.\n"
        "Z-Image Turbo: негатив не работает (CFG=1), управляй позитивом.")
    m = gr.Radio(list(MODELS.keys()), value=list(MODELS.keys())[0], label="Модель")
    p = gr.Textbox(label="Промт (позитив)", lines=3)
    n = gr.Textbox(label="Негатив (только для WAI)", lines=2)
    with gr.Row():
        s = gr.Slider(4, 50, value=8, step=1, label="Шаги")
        c = gr.Slider(1, 12, value=1.0, step=0.5, label="CFG")
        w = gr.Slider(512, 1024, value=768, step=64, label="Ширина")
        h = gr.Slider(512, 1024, value=768, step=64, label="Высота")
        sd = gr.Number(value=-1, precision=0, label="Сид (-1 = случайный)")
    btn = gr.Button("Сгенерировать")
    out = gr.Image(type="pil", label="Результат (скачивание - правой кнопкой)")
    info = gr.Textbox(label="Сид этой картинки", interactive=False)
    btn.click(gen, [m, p, n, s, c, w, h, sd], [out, info])

demo.queue(max_size=4)
demo.launch(share=True)
